In [5]:
import os, json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://us.api.openai.com/v1"
)
PROMPT_PATH = Path("../prompts/old_prompt.txt")

OUTPUT_DIR = Path("../data/model_eval/gpt_batch")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(PROMPT_PATH, "r") as f:
    PROMPT_TEMPLATE = f.read()

VALID_LABELS = [
    "unstoppable_force",
    "future_shaper",
    "natural_disaster",
    "human_replacement",
    "junk_food",
    "weapon_threat",
    "thief",
    "helper_tool",
    "knowledge_retrieval",
    "brain_mind",
    "machine_robot_system",
    "mirror_echo",
    "child",
    "synthesizer_creator",
    "friend",
    "living_creature",
    "unexplored_realm",
    "god",
    "genie_folklore",
]


MODEL = "gpt-4.1"

GPT_PRICES_PER_1M = {
    "gpt-4.1": {"input": 2.00, "output": 8.00},
}

def build_prompt(text):
    return PROMPT_TEMPLATE.replace("{text}", str(text))


def extract_dominant_from_output(raw_output):
    try:
        parsed = json.loads(raw_output)
        counts = parsed.get("metaphor_counts", {})

        if not counts:
            return "None"

        dominant = max(counts.items(), key=lambda x: x[1])[0]

        if counts[dominant] == 0:
            return "None"

        return dominant

    except:
        return "ERROR"


def make_batch_file(df, out_path):
    with open(out_path, "w") as f:
        for i, row in df.reset_index(drop=True).iterrows():

            request = {
                "custom_id": f"{MODEL}__row_{i}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "temperature": 0,
                    "max_tokens": 200,  
                    "messages": [
                        {"role": "system", "content": "Return valid JSON only."},
                        {"role": "user", "content": build_prompt(row["ai_metaphor"])},
                    ],
                },
            }

            f.write(json.dumps(request) + "\n")


def submit_batch(batch_file_path):
    uploaded = client.files.create(
        file=open(batch_file_path, "rb"),
        purpose="batch",
    )

    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    return batch

In [11]:
df = pd.read_csv('../Data/LLM_Comparison/Metaphor_Data_Myra_Chang.csv')

In [12]:
batch_file = OUTPUT_DIR / "gpt41_old_prompt.jsonl"

make_batch_file(df, batch_file)

batch = submit_batch(batch_file)

print("Batch ID:", batch.id)
print("Status:", batch.status)

Batch ID: batch_69ebbd937e54819084fd831c817610d1
Status: validating


In [13]:
batch = client.batches.retrieve(batch.id)
print(batch.status)
print(batch.request_counts)

validating
BatchRequestCounts(completed=0, failed=0, total=0)
